# 🔵🔴 Binary Classification using Feedforward Neural Networks

In this notebook, we solve a **binary classification problem** — deciding whether an input belongs to class 0 or class 1 — using two approaches:

1. **NumPy from scratch** — manual forward pass, binary cross-entropy loss, and backpropagation with sigmoid output.
2. **TensorFlow/Keras** — the same problem solved concisely using the high-level API.

---

### 🧠 What changes compared to Regression?

The network architecture is very similar, but two things change:

- **Output activation**: We use **Sigmoid** instead of linear. Sigmoid squashes any number into the range (0, 1), which we interpret as a *probability*.
- **Loss function**: We use **Binary Cross-Entropy** instead of MSE. It penalizes confident wrong predictions very heavily.

**Dataset:** We use a circular (non-linearly separable) dataset — the classic "moons" or "circles" problem that a simple linear model cannot solve.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

np.random.seed(42)

## 📦 Step 1: Generate the Dataset

We'll use `make_moons` — a classic benchmark where the two classes form interleaved crescent shapes. A linear model (like logistic regression) will struggle here; a neural network with hidden layers can easily learn the curved decision boundary.

In [ ]:
# Generate moons dataset — two interleaved half-circles
X, y = make_moons(n_samples=500, noise=0.25, random_state=42)
y = y.reshape(-1, 1)  # Shape: (500, 1)

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Visualize
plt.figure(figsize=(7, 5))
plt.scatter(X[y.flatten()==0, 0], X[y.flatten()==0, 1], s=20, label='Class 0', alpha=0.7, color='steelblue')
plt.scatter(X[y.flatten()==1, 0], X[y.flatten()==1, 1], s=20, label='Class 1', alpha=0.7, color='coral')
plt.title('Binary Classification Dataset: Two Moons')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 🔧 Part 1: Neural Network from Scratch (NumPy)

Architecture:
```
Input (2) → Hidden Layer (16, ReLU) → Hidden Layer (8, ReLU) → Output (1, Sigmoid)
```

### The Sigmoid Function

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

It maps any real number to (0, 1). Output > 0.5 → Class 1, else → Class 0.

### Binary Cross-Entropy Loss

$$L = -\frac{1}{n} \sum \left[ y \log(\hat{y}) + (1-y) \log(1-\hat{y}) \right]$$

This loss is 0 when predictions are perfect, and grows rapidly when you are confidently wrong.

In [ ]:
class BinaryClassifierNumpy:
    """
    2-hidden-layer feedforward network for binary classification.
    Uses ReLU hidden activations and Sigmoid output.
    Loss: Binary Cross-Entropy.
    """

    def __init__(self, input_dim, hidden1, hidden2, lr=0.01):
        self.lr = lr

        # He initialization for layers with ReLU
        self.W1 = np.random.randn(input_dim, hidden1) * np.sqrt(2 / input_dim)
        self.b1 = np.zeros((1, hidden1))
        self.W2 = np.random.randn(hidden1, hidden2) * np.sqrt(2 / hidden1)
        self.b2 = np.zeros((1, hidden2))
        self.W3 = np.random.randn(hidden2, 1) * np.sqrt(2 / hidden2)
        self.b3 = np.zeros((1, 1))

    def relu(self, z):
        return np.maximum(0, z)

    def relu_derivative(self, z):
        return (z > 0).astype(float)

    def sigmoid(self, z):
        """Numerically stable sigmoid."""
        return np.where(z >= 0,
                        1 / (1 + np.exp(-z)),
                        np.exp(z) / (1 + np.exp(z)))

    def forward(self, X):
        # Hidden layer 1
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.relu(self.z1)

        # Hidden layer 2
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self.relu(self.z2)

        # Output layer with Sigmoid activation
        self.z3 = self.a2 @ self.W3 + self.b3
        self.a3 = self.sigmoid(self.z3)
        return self.a3

    def compute_loss(self, y_pred, y_true):
        """Binary cross-entropy loss — clipped for numerical stability."""
        eps = 1e-15
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    def backward(self, X, y_true, y_pred):
        n = X.shape[0]

        # Gradient of BCE loss + sigmoid combined simplifies to (y_pred - y_true) / n
        dL_dz3 = (y_pred - y_true) / n

        dW3 = self.a2.T @ dL_dz3
        db3 = np.sum(dL_dz3, axis=0, keepdims=True)

        dL_da2 = dL_dz3 @ self.W3.T
        dL_dz2 = dL_da2 * self.relu_derivative(self.z2)
        dW2 = self.a1.T @ dL_dz2
        db2 = np.sum(dL_dz2, axis=0, keepdims=True)

        dL_da1 = dL_dz2 @ self.W2.T
        dL_dz1 = dL_da1 * self.relu_derivative(self.z1)
        dW1 = X.T @ dL_dz1
        db1 = np.sum(dL_dz1, axis=0, keepdims=True)

        # Update weights
        self.W3 -= self.lr * dW3;  self.b3 -= self.lr * db3
        self.W2 -= self.lr * dW2;  self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1;  self.b1 -= self.lr * db1

    def train(self, X, y, epochs=500):
        history = []
        for epoch in range(epochs):
            y_pred = self.forward(X)
            loss   = self.compute_loss(y_pred, y)
            self.backward(X, y, y_pred)
            history.append(loss)
            if (epoch + 1) % 100 == 0:
                acc = np.mean((y_pred > 0.5).astype(int) == y)
                print(f"Epoch {epoch+1}/{epochs} — Loss: {loss:.4f} | Accuracy: {acc:.4f}")
        return history

    def predict_proba(self, X):
        return self.forward(X)

    def predict(self, X):
        return (self.predict_proba(X) > 0.5).astype(int)

In [ ]:
model_np = BinaryClassifierNumpy(input_dim=2, hidden1=16, hidden2=8, lr=0.05)
history_np = model_np.train(X_train_s, y_train, epochs=1000)

y_pred_np = model_np.predict(X_test_s)
acc_np = accuracy_score(y_test, y_pred_np)
print(f"\n[NumPy Model] Test Accuracy: {acc_np:.4f}")

---
## 🚀 Part 2: Binary Classification with TensorFlow/Keras

In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)

model_keras = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(2,)),
    keras.layers.Dense(8,  activation='relu'),
    keras.layers.Dense(1,  activation='sigmoid')  # Sigmoid for binary output
])

model_keras.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_keras = model_keras.fit(
    X_train_s, y_train,
    epochs=200, batch_size=32,
    validation_split=0.1, verbose=0
)

y_pred_keras = (model_keras.predict(X_test_s) > 0.5).astype(int)
acc_keras = accuracy_score(y_test, y_pred_keras)
print(f"[Keras Model] Test Accuracy: {acc_keras:.4f}")

---
## 📊 Step 3: Visualize Decision Boundaries & Confusion Matrices

In [ ]:
def plot_decision_boundary(model, X, y, title, ax, is_keras=False):
    """Plot the decision boundary of a classifier over a 2D grid."""
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]

    if is_keras:
        Z = (model.predict(grid, verbose=0) > 0.5).astype(int).ravel()
    else:
        Z = model.predict(grid).ravel()

    Z = Z.reshape(xx.shape)
    cmap_light = ListedColormap(['#AED6F1', '#F1948A'])
    cmap_bold  = ['steelblue', 'coral']

    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
    for cls, color in zip([0, 1], cmap_bold):
        idx = y.flatten() == cls
        ax.scatter(X[idx, 0], X[idx, 1], c=color, s=15, label=f'Class {cls}', edgecolors='k', linewidths=0.3)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_decision_boundary(model_np,    X_test_s, y_test, 'NumPy NN — Decision Boundary',  axes[0], is_keras=False)
plot_decision_boundary(model_keras, X_test_s, y_test, 'Keras NN — Decision Boundary',  axes[1], is_keras=True)
plt.suptitle('Binary Classification: Decision Boundaries', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, preds, title in zip(
    axes,
    [y_pred_np, y_pred_keras],
    ['NumPy NN', 'Keras NN']
):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred 0','Pred 1'],
                yticklabels=['True 0','True 1'])
    ax.set_title(f'{title} — Confusion Matrix')

plt.tight_layout()
plt.show()

print("=== NumPy Model Report ===")
print(classification_report(y_test, y_pred_np))
print("=== Keras Model Report ===")
print(classification_report(y_test, y_pred_keras))

---

## 🧾 Summary

In this notebook, we:

- Adapted the feedforward network from regression to **binary classification** by changing the output activation to **Sigmoid** and the loss to **Binary Cross-Entropy**.
- Both models learned a curved, non-linear decision boundary on the moons dataset — something a linear model could not do.
- The **confusion matrix** shows exactly where the models make mistakes — a valuable diagnostic tool.

**Key Takeaways:**
- Sigmoid output gives you a probability. Threshold at 0.5 to get a class label.
- Binary Cross-Entropy is the right loss for binary classification — not MSE.
- The gradient of BCE + sigmoid has a beautiful simplification: `(y_pred - y_true) / n`.

👉 **Next**: See `03_multiclass_classification/` to extend this to more than 2 classes using Softmax.